In [3]:
import pandas as pd
from pathlib import Path
import openpyxl
import shutil

In [4]:
folder = Path("/Users/d.p.saakes/Downloads/export-ddw-2025-submission-2025-06-16T14/")
xls_file = folder / "export-ddw-2025-submission-2025-06-16T14.12.55.xlsx"
target_folder = Path("./build")

# print(target_folder.resolve())


In [5]:

df = pd.read_excel(xls_file)

# add_index starting with 100
df['submission id'] = df.index + 100

df.head()

# remove rows of desk reject


/Users/d.p.saakes/code/4TU_generate_svg_from_submissions/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,Form location,When,IP country code,Project Title,Abstract,Authors and Affiliations,Submitter Name,Submitter Email,Submitter Gmail,Submitter Affiliation,...,Presentation Format,what does the audience experience?,Visualisation of the audience experience (optional),Visualisation of the audience experience (optional) - path,Documentation,Documentation - path,Link to Online Documentation,Link to Video,additional info,submission id
0,https://www.4tu.nl/du/submit/ddw-2025/ddw-2025...,2025-06-03 21:53:14.879,NL,Leaving a Trace: Ecological Thinking through S...,The ecological impact of the ruins of the Anth...,Youp Ferket,Youp,y.m.e.ferket@student.tue.nl,yferket@gmail.com,Eindhoven University of Technology,...,Exhibition; E-Magazine; Dialogues,The outcome of the project presents a few oppo...,"IMG_5164.JPG (2864 KB, image/jpeg)",uploads/000002-R-img-5164.jpg,M22_Final report_Ferket_YME_Leaving a Trace.pd...,uploads/000002-T-m22-final-report-ferket-yme-l...,NaN,NaN,Despite having partially the same title as a p...,100
1,https://www.4tu.nl/du/submit/ddw-2025/ddw-2025...,2025-06-05 11:43:33.006,NL,Social Art_ Digital Art Design to Enhance Soci...,Family is the most important emotional bond fo...,"Rui Wang, Tianqin Lu, Jun Hu / Eindhoven Unive...",Rui Wang,r.wang1@tue.nl,NaN,Eindhoven University of Technology,...,Exhibition; E-Magazine; Dialogues,Social Art will be hanged on in the living roo...,"Picture user experience.jpg (805 KB, image/jpeg)",uploads/000003-R-picture-user-experience.jpg,"Picture project images .jpg (489 KB, image/jpeg)",uploads/000003-T-picture-project-images-.jpg,NaN,NaN,NaN,101
2,https://www.4tu.nl/du/submit/ddw-2025/ddw-2025...,2025-06-05 15:42:03.812,NL,Beyond The Blade,The rapid expansion of wind energy over the la...,Authors\r\rJ.K.R. Pupping Industrial Design En...,Jesse Pupping,jesse.pupping@gmail.com,jesse.pupping@gmail.com,Delft University of Technology,...,Exhibition; E-Magazine,A video and prototype of the 'modular pumptrac...,NaN,NaN,NaN,NaN,https://resolver.tudelft.nl/uuid:deade556-7e47...,https://youtu.be/5OLoWXiQiFY?si=qxn4O6YRP98Zcb_e,NaN,102
3,https://www.4tu.nl/du/submit/ddw-2025/ddw-2025...,2025-06-09 11:07:06.435,NL,FROM WASTE TO LIFE: Exploring Bio-Affordances...,This project investigates the bio-affordances ...,NaN,Fengyuan Wang,f.wang@student.tue.nl,NaN,Eindhoven University of Technology,...,Exhibition; E-Magazine; Dialogues,The material tangibility of this project lies ...,NaN,NaN,"Poster & Booklet.pdf (11599 KB, application/pdf)",uploads/000005-T-poster-booklet.pdf,NaN,https://vimeo.com/1091745306?share=copy,NaN,103
4,https://www.4tu.nl/du/submit/ddw-2025/ddw-2025...,2025-06-09 12:17:28.011,NL,OOO (Out Of Office),OOO (Out of Office) is a discursive design pro...,NaN,Tongbin Qi,qitongbin4@gmail.com,NaN,Eindhoven University of Technology,...,Exhibition; E-Magazine; Dialogues,The audience will explore the exhibition space...,NaN,NaN,"oooddw.pdf (8605 KB, application/pdf)",uploads/000006-T-oooddw.pdf,NaN,https://vimeo.com/1046719238?share=copy,NaN,104


In [11]:
# GENERATE CSV

doc = ""

themes_names = ["Thriving Planet", "Living Environments", "Digital Future", "Health and Wellbeing", "Equal Society"]


for index, row in df.iterrows():
    line = ""
    line += str(row["submission id"])
    line += "," + row['Project Title']
    line += "," + row['Submitter Affiliation']
    line += "," + row['Submitter Name']
    line += "," + row['Submitter Email']

    themes = row["DU Theme"].split("; ")
    for t in themes_names:
        if t in themes:
            line += f",{t}"
        else:
            line += ", "
            

    line += "," + row['Presentation Format']    

    doc += line + "\n"


with open("submissions.csv", "w") as text_file:
    text_file.write(doc)



In [25]:
# make markdown documents

for index, row in df.iterrows():
    submission_id = row['submission id']
    title = row['Project Title']

    authors = row['Submitter Name']
    affiliation = row['Submitter Affiliation']
    abstract_text = row['Abstract']
    theme_fit = row['How does the project fit the theme(s) and the overall theme: "Less Hope, More Action!']
    audience_experience_paragraph = row['what does the audience experience?']
    additional_info_paragraph = row['additional info']

    link_to_online_documentation = row['Link to Online Documentation']
    link_to_video = row['Link to Video']

    md = f"submission **{submission_id}** from **{authors}** at {affiliation}\n\n"        
    md += f"# {title}\n\n"    
    md += abstract_text + "\n\n"
    md += f"## audience experience\n\n{audience_experience_paragraph}\n\n"
    md += f"## How does the project fit the theme(s)\n\n{theme_fit}\n\n"

    if type(link_to_online_documentation) == str:
        md += f"## online documentation\n\n[online documentation]({link_to_online_documentation})\n\n\n"
    if type(link_to_video) == str:
        md += f"## link to video\n\n[link to video]({link_to_video})\n\n\n"

    with open(f"{submission_id}.md", "w") as text_file:
        text_file.write(md)

    




In [21]:
# COPY FILES

for index, row in df.iterrows():

    # make folder
    submission_id = row["submission id"]
    submission_folder = target_folder / str(submission_id)
    submission_folder.mkdir(parents=True, exist_ok=True) 

    # copy files
    print(submission_id)

    doc_path = row["Documentation - path"]
    if type(doc_path) == str:
        print(f"\t{doc_path}")
        p = folder / doc_path
        shutil.copy(p, submission_folder)

    pic_path = row["Visualisation of the audience experience (optional) - path"]
    if type(pic_path) == str:
        print(f"\t{pic_path}")
        p = folder / pic_path
        shutil.copy(p, submission_folder)
    
    
    # print(f"Index: {index}\n\t title: {row['Project Title']}\n\t Institute: {row['Submitter Affiliation']}", {row["DU Theme"]},)

100
	uploads/000002-T-m22-final-report-ferket-yme-leaving-a-trace.pdf
	uploads/000002-R-img-5164.jpg
101
	uploads/000003-T-picture-project-images-.jpg
	uploads/000003-R-picture-user-experience.jpg
102
103
	uploads/000005-T-poster-booklet.pdf
104
	uploads/000006-T-oooddw.pdf
105
	uploads/000007-T-diyasamit-thesis.pdf
	uploads/000007-R-soilhabitability-interaction-speculation-setup.jpg
106
	uploads/000008-R-img-5482.jpeg
107
	uploads/000009-T-green-light-report-isabella-maljers.docx
	uploads/000009-R-img-5482.jpeg
108
	uploads/000010-T-m22-final-report-starmans-d-pixel-portraits.pdf
	uploads/000010-R-20250606-224350.jpg
109
	uploads/000011-R-5f12af4253a941eae6594401c4bf050.jpg
110
	uploads/000012-T-ddw2025-freehabilitation.pdf
	uploads/000012-R-freehab.jpg
111
	uploads/000013-T-booklet-group-6-ecological-pest-management.pdf
	uploads/000013-R-figure-4.png
112
	uploads/000014-T-designinghapticinterfacetosupporttacitknowledgeacquisition.pdf
	uploads/000014-R-img-5946.jpg
113
	uploads/000015